# Proyecto 2 FYHBD | Persona 2 — Capa Silver (`silver_transform.ipynb`)

Ejecución **manual de respaldo** (si falla Airflow). Usa exactamente la misma lógica que `scripts/silver_transform.py`.

```
Bronze (Parquet en MinIO) → PySpark (limpieza · normalización · deduplicación) → MERGE → Silver Iceberg (Nessie) → Trino
```

| Tabla Silver | Se construye desde | Clave MERGE |
|---|---|---|
| `nessie.silver.users_hist` | `bronze/users/2020` + `bronze/users/2021` | `id` |
| `nessie.silver.post_hist` | `bronze/post/2020` + `bronze/post/2021` | `id` |

Silver usa **Iceberg + MERGE (no overwrite)**, es histórica (2020 + 2021) y tiene `fecha_cargue`.

**Modo de ejecución:** Spark corre *dentro de Jupyter* (`local[2]`), sin depender del Worker. La evidencia del cluster (Master + Worker) está en los `spark-submit` de `docs/02_silver_spark_trino.md`.
Requiere Bronze cargado y `docker compose up -d minio nessie jupyter`.

## 1. Parámetros

In [1]:
TABLA = "all"               # users_hist | post_hist | all
YEARS = (2020, 2021)        # anios de Bronze a consolidar
LIMIT = None                # p.ej. 200000 para limitar filas por archivo
MASTER = "local[2]"         # Spark dentro de Jupyter (no usa el Worker del cluster)

## 2. Importar el módulo de Silver y abrir la sesión Spark (modo local)

In [2]:
import os, sys

# Forzar modo local: ignora cualquier configuracion de cluster del contenedor
os.environ["SPARK_MASTER_URL"] = MASTER
os.environ.pop("SPARK_DRIVER_HOST", None)
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 2g pyspark-shell")

sys.path.insert(0, "/workspace/scripts")   # en el contenedor Jupyter
sys.path.insert(0, "../scripts")           # por si se corre fuera de Docker
import silver_transform as s

try:
    spark.stop()                           # cierra una sesion anterior si existe
except Exception:
    pass

spark = s.get_spark(MASTER, app_name="silver_notebook")
print("Spark", spark.version, "| master:", spark.sparkContext.master)
print("Catalogo Nessie:", spark.conf.get("spark.sql.catalog.nessie.uri"))
assert spark.sparkContext.master.startswith("local"), "El notebook debe correr en modo local"


/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/12 17:37:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 | master: local[2]
Catalogo Nessie: http://nessie:19120/iceberg


## 3. Transformación Silver con MERGE (equivale a la tarea Silver de Airflow)

In [3]:
tablas = ["users_hist", "post_hist"] if TABLA == "all" else [TABLA]
resumenes = [s.run_silver(t, YEARS, LIMIT, spark=spark) for t in tablas]

26/10/12 17:37:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


[bronze] leido s3a://bronze/users/2020/users_2020.parquet
[bronze] leido s3a://bronze/users/2021/users_2021.parquet


[OK] {"tabla": "nessie.silver.users_hist", "archivos_bronze": ["s3a://bronze/users/2020/users_2020.parquet", "s3a://bronze/users/2021/users_2021.parquet"], "limit": null, "tabla_creada_en_esta_corrida": false, "filas_fuente_deduplicadas": 400000, "filas_antes": 400000, "filas_despues": 400000, "filas_nuevas": 0, "duplicados_id": 0, "filas_por_anio_creacion": {"2020": 200000, "2021": 200000}, "ultimo_snapshot": {"snapshot_id": 2111963117048288436, "committed_at": "2026-10-12 17:37:33.669000", "operation": "overwrite", "summary": {"engine-version": "3.5.3", "added-data-files": "2", "total-equality-deletes": "0", "app-id": "local-1791844633142", "added-records": "400000", "deleted-data-files": "2", "deleted-records": "400000", "total-records": "400000", "spark.app.id": "local-1791844633142", "removed-files-size": "26701185", "changed-partition-count": "2", "engine-name": "spark", "total-position-deletes": "0", "added-files-size": "26701185", "total-delete-files": "0", "iceberg-version": "

26/10/12 17:37:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

[OK] {"tabla": "nessie.silver.post_hist", "archivos_bronze": ["s3a://bronze/post/2020/post_2020.parquet", "s3a://bronze/post/2021/post_2021.parquet"], "limit": null, "tabla_creada_en_esta_corrida": false, "filas_fuente_deduplicadas": 400000, "filas_antes": 400000, "filas_despues": 400000, "filas_nuevas": 0, "duplicados_id": 0, "filas_por_anio_creacion": {"2020": 200000, "2021": 200000}, "ultimo_snapshot": {"snapshot_id": 2655974010588836228, "committed_at": "2026-10-12 17:37:56.709000", "operation": "overwrite", "summary": {"engine-version": "3.5.3", "added-data-files": "2", "total-equality-deletes": "0", "app-id": "local-1791844633142", "added-records": "400000", "deleted-data-files": "2", "deleted-records": "400000", "total-records": "400000", "spark.app.id": "local-1791844633142", "removed-files-size": "32061035", "changed-partition-count": "2", "engine-name": "spark", "total-position-deletes": "0", "added-files-size": "32061035", "total-delete-files": "0", "iceberg-version": "Apach

## 4. Verificación en Iceberg (consultas vía Spark)

In [4]:
for t in tablas:
    fq = f"nessie.silver.{t}"
    print("==", fq)
    spark.sql(f"SELECT creation_year, COUNT(*) AS filas, MAX(fecha_cargue) AS ultimo_cargue FROM {fq} GROUP BY 1 ORDER BY 1").show()
    spark.sql(f"SELECT committed_at, operation, summary['added-records'] AS agregados, summary['total-records'] AS total "
              f"FROM {fq}.snapshots ORDER BY committed_at").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.silver").show()

== nessie.silver.users_hist
+-------------+------+-------------------+
|creation_year| filas|      ultimo_cargue|
+-------------+------+-------------------+
|         2020|200000|2026-10-12 22:06:21|
|         2021|200000|2026-10-12 22:06:21|
+-------------+------+-------------------+

+-----------------------+---------+---------+------+
|committed_at           |operation|agregados|total |
+-----------------------+---------+---------+------+
|2026-10-12 22:37:33.669|overwrite|400000   |400000|
+-----------------------+---------+---------+------+

== nessie.silver.post_hist
+-------------+------+-------------------+
|creation_year| filas|      ultimo_cargue|
+-------------+------+-------------------+
|         2020|200000|2026-10-12 22:07:36|
|         2021|200000|2026-10-12 22:07:36|
+-------------+------+-------------------+

+-----------------------+---------+---------+------+
|committed_at           |operation|agregados|total |
+-----------------------+---------+---------+------+
|2

## 5. Idempotencia del MERGE (evidencia)
Re-ejecutar **no duplica** filas: `filas_nuevas` debe ser 0 y `duplicados_id` 0.

In [5]:
rep = [s.run_silver(t, YEARS, LIMIT, spark=spark) for t in tablas]
for r in rep:
    print(r["tabla"], "| filas_antes:", r["filas_antes"], "| filas_despues:", r["filas_despues"],
          "| filas_nuevas:", r["filas_nuevas"], "| duplicados_id:", r["duplicados_id"])

[bronze] leido s3a://bronze/users/2020/users_2020.parquet
[bronze] leido s3a://bronze/users/2021/users_2021.parquet


[OK] {"tabla": "nessie.silver.users_hist", "archivos_bronze": ["s3a://bronze/users/2020/users_2020.parquet", "s3a://bronze/users/2021/users_2021.parquet"], "limit": null, "tabla_creada_en_esta_corrida": false, "filas_fuente_deduplicadas": 400000, "filas_antes": 400000, "filas_despues": 400000, "filas_nuevas": 0, "duplicados_id": 0, "filas_por_anio_creacion": {"2020": 200000, "2021": 200000}, "ultimo_snapshot": {"snapshot_id": 87154786244213462, "committed_at": "2026-10-12 17:38:09.681000", "operation": "overwrite", "summary": {"engine-version": "3.5.3", "added-data-files": "2", "total-equality-deletes": "0", "app-id": "local-1791844633142", "added-records": "400000", "deleted-data-files": "2", "deleted-records": "400000", "total-records": "400000", "spark.app.id": "local-1791844633142", "removed-files-size": "26701185", "changed-partition-count": "2", "engine-name": "spark", "total-position-deletes": "0", "added-files-size": "26701185", "total-delete-files": "0", "iceberg-version": "Ap

[OK] {"tabla": "nessie.silver.post_hist", "archivos_bronze": ["s3a://bronze/post/2020/post_2020.parquet", "s3a://bronze/post/2021/post_2021.parquet"], "limit": null, "tabla_creada_en_esta_corrida": false, "filas_fuente_deduplicadas": 400000, "filas_antes": 400000, "filas_despues": 400000, "filas_nuevas": 0, "duplicados_id": 0, "filas_por_anio_creacion": {"2020": 200000, "2021": 200000}, "ultimo_snapshot": {"snapshot_id": 7007833947660555833, "committed_at": "2026-10-12 17:38:22.174000", "operation": "overwrite", "summary": {"engine-version": "3.5.3", "added-data-files": "2", "total-equality-deletes": "0", "app-id": "local-1791844633142", "added-records": "400000", "deleted-data-files": "2", "deleted-records": "400000", "total-records": "400000", "spark.app.id": "local-1791844633142", "removed-files-size": "32061035", "changed-partition-count": "2", "engine-name": "spark", "total-position-deletes": "0", "added-files-size": "32061035", "total-delete-files": "0", "iceberg-version": "Apach

In [6]:
spark.stop()